# Markdown & Clearance Pricing Optimization for Multi-SKU Retail Inventory

**Optimization and Simulation — Final Project**

This notebook documents the end-to-end pipeline for a Mixed-Integer Programming (MIP) model that recommends a week-by-week markdown schedule for retail SKUs, maximizing recovered revenue while clearing inventory by a fixed deadline. The full implementation lives in `src/`; this notebook walks through the data, methodology, key modeling decisions, and results, and loads pre-computed outputs rather than re-running the full solve (which takes ~18.5 minutes across all departments).

**Data source:** [M5 Forecasting Accuracy](https://www.kaggle.com/competitions/m5-forecasting-accuracy/data) (Walmart, public).

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.dpi'] = 110

PROCESSED_DIR = Path('data/processed')
OUTPUTS_DIR = Path('outputs')

## 1. Data Description

M5 does not include real inventory levels or unit costs — these are derived as documented modeling assumptions (Section 4). The three raw files used are `sell_prices.csv`, `sales_train_evaluation.csv`, and `calendar.csv`; the full data prep pipeline (`src/data_prep.py`) melts the wide sales file to long format, joins prices and calendar context, and aggregates to a SKU-week panel.

In [ ]:
panel = pd.read_csv(PROCESSED_DIR / 'sku_week_panel.csv')

print(f"Total SKU-store-week rows: {len(panel):,}")
print(f"Categories: {panel['cat_id'].unique().tolist()}")
print(f"Departments: {sorted(panel['dept_id'].unique().tolist())}")
print(f"Unique items: {panel['item_id'].nunique():,}")
print(f"Stores: {panel['store_id'].nunique()}")
panel.head()

In [ ]:
dept_counts = panel.groupby('dept_id')['item_id'].nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
dept_counts.plot(kind='bar', ax=ax, color='#4C72B0')
ax.set_ylabel('Unique items')
ax.set_title('Items per department (full catalog, before filtering)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 2. Problem Formulation

The markdown decision is formulated as a **Mixed-Integer Program**:

- **Objective:** maximize total recovered revenue across all SKUs and weeks in the clearance horizon.
- **Decision variables:** a binary selection, per SKU per week, of a discount tier from a discrete price ladder (0%, 10%, 20%, 30%, 50% off), plus continuous units-sold variables gated to each tier.
- **Constraints:**
  1. Inventory conservation — cumulative units sold ≤ starting stock, per SKU
  2. End-of-horizon clearance — ≥95% of units sold by the deadline
  3. Monotonic pricing — discounts cannot decrease week over week
  4. Margin floor — price cannot drop below cost + minimum margin, except a designated final emergency-clearance week

Because the discount ladder is already discrete, demand at each tier is a known constant looked up from the SKU's fitted price-response curve — this keeps the model perfectly linear (a *tier-disaggregated* MIP) with no bilinear price × units-sold term, rather than requiring true SOS2 interpolation. Full implementation: `src/model.py`.

## 3. Price-Elasticity Estimation

For each item-store pair, we fit:

$$\log(\text{units\_sold} + 1) = \beta_0 + \beta_1 \log(\text{price}) + \beta_2 \cdot \text{has\_event} + \varepsilon$$

via closed-form OLS (`src/elasticity.py`). Item-store pairs with fewer than 10 observations fall back to a department-pooled estimate. $\beta_1$ is the price elasticity of demand.

In [ ]:
elasticities = pd.read_csv(PROCESSED_DIR / 'elasticities.csv')

print(f"Fitted {len(elasticities):,} item-store elasticity curves")
print(f"  item_store-level fits: {(elasticities['fit_level'] == 'item_store').sum():,}")
print(f"  dept_pooled fallbacks: {(elasticities['fit_level'] == 'dept_pooled').sum():,}")
elasticities[['elasticity', 'r2']].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(elasticities['r2'], bins=40, color='#55A868')
axes[0].axvline(0.10, color='red', linestyle='--', label='R² = 0.10 threshold')
axes[0].set_xlabel('R²')
axes[0].set_title('Distribution of fit quality (R²)')
axes[0].legend()

axes[1].hist(elasticities['elasticity'], bins=60, color='#C44E52')
axes[1].axvline(0, color='black', linestyle='--', label='elasticity = 0')
axes[1].set_xlabel('Fitted elasticity')
axes[1].set_title('Distribution of fitted elasticity values')
axes[1].legend()

plt.tight_layout()
plt.show()

### A key methodological finding

The median R² across all fitted curves is low (~0.05), reflecting that most SKUs in M5 have limited price variation to estimate elasticity from. More importantly, a subset of low-R² fits produced **economically implausible positive elasticities** — implying demand *rises* with price, which is backwards for a normal good and a symptom of overfitting sparse, noisy data rather than a genuine relationship.

Including these SKUs made department-level MIPs infeasible outright: a SKU whose fitted curve predicts near-zero demand at every price cannot plausibly clear 95% of its assumed inventory, and it only takes one such SKU to sink an entire department's joint solve. Rather than treat this as a bug to hide, we treat it as a documented scoping decision (`src/elasticity_filter.py`): SKUs are excluded if they fail any of —

1. **R² ≥ 0.10** (minimally reliable fit)
2. **Elasticity < 0 and ≥ -10** (economically sensible sign and magnitude)
3. **Structural feasibility** — the SKU's own best-case demand curve must be able to sell ≥95% of its assumed starting inventory within its horizon

In [ ]:
with open(OUTPUTS_DIR / 'results_summary.json') as f:
    results = json.load(f)

# Filtering stats (re-derived here for display; see src/elasticity_filter.py for the source of truth)
total_skus = 30490
excluded_low_r2 = 22658
excluded_bad_elasticity = 2998
excluded_structural = 0
kept = 4834

filter_stats = pd.Series({
    'Excluded: R² < 0.10': excluded_low_r2,
    'Excluded: implausible elasticity sign/magnitude': excluded_bad_elasticity,
    'Excluded: structurally infeasible': excluded_structural,
    'Kept for modeling': kept,
})

fig, ax = plt.subplots(figsize=(7, 4))
filter_stats.plot(kind='barh', ax=ax, color=['#C44E52', '#DD8452', '#8C8C8C', '#55A868'])
ax.set_xlabel('Number of SKUs')
ax.set_title(f'SKU filtering funnel ({total_skus:,} total → {kept:,} kept, {kept/total_skus:.1%})')
plt.tight_layout()
plt.show()

## 4. Modeling Assumptions: Inventory & Cost

M5 provides no real inventory levels or unit costs, since retailers don't publish these alongside a public forecasting dataset. Both are derived as transparent, documented assumptions (`src/inventory_assumptions.py`):

- **Starting inventory** = 4 weeks-of-supply × the SKU's own average weekly units sold (a standard retail sizing heuristic, scaled to the SKU's real demand rather than an arbitrary constant), with a floor of 5 units.
- **Unit cost** = 60% of base price (a 40% assumed margin).

These are stated explicitly as modeling choices rather than presented as real data.

In [ ]:
model_ready = pd.read_csv(PROCESSED_DIR / 'model_ready_panel_filtered.csv')
sku_level = model_ready.drop_duplicates(['item_id', 'store_id']).dropna(subset=['starting_inventory'])

print(f"Modeled SKU-store pairs: {len(sku_level):,}")
sku_level[['starting_inventory', 'unit_cost', 'base_price', 'elasticity', 'r2']].describe()

## 5. Hybrid Decomposition: Solving at Scale

Solving all SKUs jointly across the full clearance horizon is computationally impractical: even a modest subset produces thousands of binary variables with piecewise-demand structure, well into the regime where MIP branch-and-bound solve time grows combinatorially. This is addressed with a three-layer hybrid decomposition (`src/decomposition.py`):

1. **Department clustering** — SKUs are grouped by `dept_id` (a natural, pre-existing grouping) and solved as independent sub-MIPs, assuming weak cross-department price interactions.
2. **SKU batching within department** — departments with more than 100 SKUs are further split into batches, since even a single department's SKU set can be too large to solve jointly within one rolling-horizon window (empirically confirmed: solve time did not scale linearly past ~100 SKUs jointly).
3. **Rolling horizon** — within a batch, an 8-week window is solved exactly, the first 4 weeks are committed using the *actual* rolled-forward inventory, and the window advances. The end-of-horizon clearance constraint is enforced only on the window that reaches the true horizon end.

Each sub-solve is exact (CBC proves optimality); the decomposition itself is the heuristic layer — trading global optimality for tractability, which is the honest, defensible trade-off this project argues for.

## 6. Results

Loading the saved output of the full solve (`outputs/results_summary.json`, `outputs/sample_markdown_schedule.csv`) rather than re-running the ~18.5 minute solve inline.

In [ ]:
print(f"Solve time: {results['solve_time_seconds']:.1f}s ({results['solve_time_seconds']/60:.1f} minutes)")
print(f"All departments feasible: {results['all_departments_feasible']}")
print()
print(f"MIP-optimized revenue:      ${results['mip_total_revenue']:,.2f}")
print(f"Flat 20% discount baseline: ${results['flat_discount_baseline_revenue']:,.2f}")
print(f"Calendar-based baseline:    ${results['calendar_baseline_revenue']:,.2f}")
print()
print(f"MIP recovers {results['pct_improvement_vs_flat']:+.1f}% vs. flat discount")
print(f"MIP recovers {results['pct_improvement_vs_calendar']:+.1f}% vs. calendar-based markdown")

In [ ]:
revenue_comparison = pd.Series({
    'Flat 20% Discount': results['flat_discount_baseline_revenue'],
    'Calendar-Based Markdown': results['calendar_baseline_revenue'],
    'MIP-Optimized (this project)': results['mip_total_revenue'],
})

fig, ax = plt.subplots(figsize=(7, 4.5))
colors = ['#8C8C8C', '#DD8452', '#55A868']
bars = ax.bar(revenue_comparison.index, revenue_comparison.values, color=colors)
ax.set_ylabel('Recovered Revenue ($)')
ax.set_title('Recovered Revenue: MIP-Optimized vs. Naive Baselines')
for bar, val in zip(bars, revenue_comparison.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 5000, f'${val:,.0f}', ha='center', fontsize=9)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
dept_status = results['per_department_status']
dept_df = pd.DataFrame([
    {'dept_id': dept, 'revenue': v['objective_value'], 'windows_solved': v['n_windows_solved'], 'feasible': v['feasible']}
    for dept, v in dept_status.items()
]).sort_values('revenue', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(dept_df['dept_id'], dept_df['revenue'], color='#4C72B0')
ax.set_ylabel('Recovered Revenue ($)')
ax.set_title('Recovered Revenue by Department')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

dept_df

### Example: one SKU's optimized markdown schedule

To make the MIP's output concrete, here's the actual week-by-week price and units-sold schedule for one sample SKU-store pair, pulled from the saved solve output. We deliberately select a SKU that shows real markdown behavior (several distinct discount tiers used, reaching a meaningfully deep discount) rather than one that stays at full price the whole horizon, since that makes for a far more illustrative example of what the model actually does.

In [ ]:
schedule = pd.read_csv(OUTPUTS_DIR / 'sample_markdown_schedule.csv')

# Find SKUs with the richest markdown story: most distinct tiers used AND a
# meaningfully deep final discount, so the trajectory is worth plotting.
sku_summary = schedule.groupby(['item_id', 'store_id']).agg(
    n_tiers=('tier', 'nunique'),
    max_tier=('tier', 'max'),
    n_weeks=('week', 'count'),
).reset_index()

candidates = sku_summary[(sku_summary['n_tiers'] >= 3) & (sku_summary['max_tier'] >= 3)]
candidates = candidates.sort_values(['n_tiers', 'max_tier'], ascending=False)

if len(candidates) > 0:
    sample_item, sample_store = candidates.iloc[0][['item_id', 'store_id']]
else:
    fallback = sku_summary.sort_values('n_tiers', ascending=False).iloc[0]
    sample_item, sample_store = fallback['item_id'], fallback['store_id']

sample = schedule[
    (schedule['item_id'] == sample_item) & (schedule['store_id'] == sample_store)
].sort_values('week').reset_index(drop=True)

# Use a sequential week index for display instead of raw wm_yr_wk codes,
# which are not meant for plotting and become unreadable at 200+ weeks
sample['week_number'] = range(1, len(sample) + 1)

print(f"Sample SKU: {sample_item} @ {sample_store} ({len(sample)} weeks, "
      f"{sample['tier'].nunique()} distinct tiers used, max tier reached: {sample['tier'].max()})")
sample[['week_number', 'tier', 'price', 'units_sold']].head(15)

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4.5))

ax1.plot(sample['week_number'], sample['price'], marker='o', markersize=3, color='#C44E52', label='Price ($)')
ax1.set_ylabel('Price ($)', color='#C44E52')
ax1.tick_params(axis='y', labelcolor='#C44E52')
ax1.set_xlabel('Week')

ax2 = ax1.twinx()
ax2.bar(sample['week_number'], sample['units_sold'], alpha=0.3, color='#4C72B0', label='Units sold')
ax2.set_ylabel('Units Sold', color='#4C72B0')
ax2.tick_params(axis='y', labelcolor='#4C72B0')

# Thin the x-axis ticks so it stays readable across a long (200+ week) horizon
n_ticks = 10
tick_positions = sample['week_number'].iloc[::max(1, len(sample) // n_ticks)]
ax1.set_xticks(tick_positions)

plt.title(f'Optimized Markdown Schedule: {sample_item} @ {sample_store}')
fig.tight_layout()
plt.show()

**A note on a bug caught via this exact chart:** an earlier version of the rolling-horizon decomposition enforced monotonic (non-increasing) pricing only *within* each rolling window, but nothing prevented price from illegally resetting back toward full price at a window boundary once a fresh window began solving. This was caught by visually inspecting a sample SKU's schedule (price appeared to *rise* partway through the horizon, which should be structurally impossible given the model's own constraints) and confirmed with a regression test asserting monotonicity holds across the full committed schedule, not just within one window. The fix carries each SKU's last *committed* discount tier forward as a floor for the next window (`src/decomposition.py`), and cost a small, honest reduction in total recovered revenue (~0.55%, from $440,633 to $438,202) -- reflecting the true cost of correctly enforcing a constraint that was previously being silently violated.

## 7. Sensitivity & Limitations

**Sensitivity considerations** (see project proposal and `src/solve.py` CLI flags for parameters that can be varied):

- **Inventory assumption (`--inventory-weeks`)**: reducing weeks-of-supply from 8 to 4 was itself a sensitivity-driven decision — it directly affects how many SKUs are structurally capable of hitting the clearance target, and a lower assumption is arguably more realistic for a SKU already headed to markdown.
- **R² / elasticity filtering thresholds**: the 15.9% SKU retention rate is sensitive to the `--min-r2` threshold in `src/elasticity_filter.py`; a stricter threshold would further shrink the modeled catalog but increase confidence in each retained SKU's demand curve.
- **SKU batch size (`--sku-batch-size`)**: trades off solve time against how much cross-SKU structure is captured jointly within a department; 100 SKUs per batch was chosen empirically after observing that larger joint solves did not scale linearly in solve time.

**Limitations, stated honestly:**

- Median elasticity fit quality is weak (R² ≈ 0.05) across the full M5 catalog, reflecting limited real price variation in the underlying data for most SKUs — this project explicitly filters to the subset of SKUs (15.9%) where the fit is reliable and economically sensible, rather than modeling the full catalog with low-confidence curves.
- Inventory and unit cost are not present in M5 and are estimated via documented heuristics (Section 4), not observed values.
- The rolling-horizon decomposition trades global optimality for tractability; each sub-solve is exact, but the overall schedule across windows is not guaranteed globally optimal.

## 8. Conclusion

The MIP-optimized markdown schedule recovers **23.6% more revenue** than a naive flat-discount policy and **12.0% more** than a calendar-based markdown schedule, across 4,834 modeled SKU-store pairs spanning all three M5 product categories. These figures reflect the corrected schedule after fixing a cross-window monotonicity issue in the rolling-horizon decomposition (Section 6), which is itself a piece of evidence that the model's constraints are being validated and honestly enforced, not just assumed correct. The full pipeline — data preparation, elasticity estimation, MIP formulation, and hybrid decomposition — runs end-to-end on CPU only in under 20 minutes, and is deployed as an interactive Streamlit app for live scenario exploration (see README for the live demo link).